# IMU Segraw → Moments Feature Engineering

Pipeline:
1. Load raw segmented IMU CSVs into nested dict
2. Skip BPF (IMU is low-frequency; BPF at 20-450 Hz would destroy it)
3. Per-gesture std normalisation (same as EMG pipeline)
4. Khushaba spectral moments FE (5 features × 90 IMU channels = 450 features per gesture)
5. Global feature-matrix std normalisation
6. Save to CSV — one row per gesture trial

Mirror of `ppdsegraw_feature_engineering.ipynb` for EMG, with two changes:
- `modalities=['I']` in `load_segraw_data`
- `already_BPFd=True` in `apply_filter_to_nested_dict` (skips BPF)

Output file: `moments_segraw_allIMU.csv`

In [1]:
import numpy as np
import pandas as pd
import time
import json

# All shared preprocessing + FE functions live here
from shared_processing import (
    load_segraw_data,
    apply_filter_to_nested_dict,
    normalize_gestures_by_std_any_channels,
    create_khushaba_spectralmomentsFE_vectors,
    normalize_whole_dataset_features,
)

## 0. Config — set your paths here

In [6]:
# ── UPDATE THESE TO YOUR MACHINE ──────────────────────────────────────────
# C:\Users\kdmen\Box\Yamagami Lab\Data\2024_UIST_dataset\upload\segmented_raw_data
RAW_DATA_DIR   = r"C:\Users\kdmen\Box\Yamagami Lab\Data\2024_UIST_dataset\upload\segmented_raw_data"
SAVE_DIR       = r"C:\Users\kdmen\Box\Yamagami Lab\Data\Meta_Gesture_Project"
# ──────────────────────────────────────────────────────────────────────────

# Intermediate save (nested dict after loading, before FE) — optional but
# saves ~5 min if you need to re-run FE without reloading CSVs
RAW_DICT_SAVE_PATH      = SAVE_DIR + r"\segraw_IMU_allgestures_allusers.json"
PPD_DICT_SAVE_PATH      = SAVE_DIR + r"\ppdsegraw_allIMU.json"
MOMENTS_CSV_SAVE_PATH   = SAVE_DIR + r"\moments_segraw_allIMU.csv"

# Participant lists — identical to your EMG pipeline
pIDs_impaired   = ['P102','P103','P104','P105','P106','P107','P108','P109','P110','P111',
                   'P112','P114','P115','P116','P118','P119','P121','P122','P123','P124',
                   'P125','P126','P127','P128','P131','P132']
pIDs_unimpaired = ['P004','P005','P006','P008','P010','P011']
pIDs_both       = pIDs_impaired + pIDs_unimpaired

# NOTE: 2000 Hz was for EMG
# 148 Hz was for IMU
FS = 148  # Nominal sampling rate stored in the raw files (used for moment calculations)

## 1. Load raw IMU CSVs

> Same `load_segraw_data` as the EMG pipeline — just pass `modalities=['I']`.
> Each gesture ends up as a list of 72 channels × ~300 timepoints.

In [ ]:
# This took 50 minutes for exp-def IMU

print("Loading raw IMU data...")
start = time.time()

nested_dict = load_segraw_data(
    pIDs      = pIDs_both,
    data_dir_path = RAW_DATA_DIR,
    modalities    = ["I"],          # ← IMU only (was ["E"] for EMG pipeline)
    expt_types    = ["experimenter-defined"],
    num_imu_sensors = 12,
)

print(f"Done in {time.time()-start:.1f}s")
print(f"Participants loaded: {list(nested_dict.keys())}")

Loading raw IMU data...
P102
P103
P104
P105
P106
P107
P108
P109
P110
P111
P112
P114
P115
P116
P118
P119
P121
P122
P123
P124
P125
P126
P127
P128
P131
P132
P004
P005
P006
P008
P010
P011
Done in 2950.2s
Participants loaded: ['P102', 'P103', 'P104', 'P105', 'P106', 'P107', 'P108', 'P109', 'P110', 'P111', 'P112', 'P114', 'P115', 'P116', 'P118', 'P119', 'P121', 'P122', 'P123', 'P124', 'P125', 'P126', 'P127', 'P128', 'P131', 'P132', 'P004', 'P005', 'P006', 'P008', 'P010', 'P011']


In [4]:
# Quick sanity check — should be 72 channels (12 sensors × 6 axes) --> It might think there's 15 sensors since the highest sensor channel was labeled 15...
sample_pid     = list(nested_dict.keys())[0]
sample_gesture = list(nested_dict[sample_pid].keys())[0]
sample_trial   = list(nested_dict[sample_pid][sample_gesture].keys())[0]
sample_data    = nested_dict[sample_pid][sample_gesture][sample_trial]["IMU"]

print(f"Sample: {sample_pid} / {sample_gesture} / trial {sample_trial}")
print(f"  n_channels  : {len(sample_data)}   (expected 72)")
print(f"  n_timepoints: {len(sample_data[0])}  (expected ~300)")

Sample: P102 / pan / trial 1
  n_channels  : 72   (expected 72)
  n_timepoints: 474  (expected ~300)


In [ ]:
# Took 5 minutes to save lol

# Optional: save the raw nested dict so you don't have to reload CSVs next time
print("Saving raw IMU nested dict...")
import json
with open(RAW_DICT_SAVE_PATH, 'w') as f:
    json.dump(nested_dict, f)
print(f"Saved → {RAW_DICT_SAVE_PATH}")

Saving raw IMU nested dict...
Saved → C:\Users\kdmen\Box\Yamagami Lab\Data\Meta_Gesture_Project\segraw_IMU_allgestures_allusers.json


## 2. Normalise — skip BPF, mean-subtract, then per-gesture std=1

> **Key difference from EMG pipeline:** `already_BPFd=True` skips the
> 20-450 Hz bandpass filter entirely.  IMU signals are low-frequency;
> that BPF would zero out almost everything.
>
> We still do mean subtraction + per-gesture std normalisation,
> which is the same as the EMG pipeline.

In [8]:
print("Mean-subtracting (no BPF)...")
start = time.time()

ms_dict = apply_filter_to_nested_dict(
    nested_dict,
    normalization_method = "MEANSUBTRACTION",
    already_BPFd         = True,   # ← skips BPF
)

print(f"Done in {time.time()-start:.1f}s")

Mean-subtracting (no BPF)...
Done in 12.0s


In [9]:
print("Per-gesture std normalisation...")
start = time.time()

# Use the any-channels variant so it handles 90 IMU channels automatically
ppd_dict = normalize_gestures_by_std_any_channels(ms_dict)

print(f"Done in {time.time()-start:.1f}s")

# Sanity check: std across flattened channels should be ~1
sample_data_ppd = ppd_dict[sample_pid][sample_gesture][sample_trial]["IMU"]
flat = [v for ch in sample_data_ppd for v in ch]
print(f"  Post-normalisation std (should be ~1.0): {np.std(flat):.4f}")

Per-gesture std normalisation...
Done in 12.8s
  Post-normalisation std (should be ~1.0): 1.0000


In [10]:
# Optional: save the preprocessed dict
print("Saving preprocessed IMU dict...")
with open(PPD_DICT_SAVE_PATH, 'w') as f:
    json.dump(ppd_dict, f)
print(f"Saved → {PPD_DICT_SAVE_PATH}")

Saving preprocessed IMU dict...
Saved → C:\Users\kdmen\Box\Yamagami Lab\Data\Meta_Gesture_Project\ppdsegraw_allIMU.json


## 3. Khushaba Moments Feature Engineering

For each gesture trial, for each of the 90 IMU channels, we compute:
- `zero_order`        — total signal power (energy)
- `second_order`      — energy of rate-of-change (first derivative)
- `fourth_order`      — energy of jerk (second derivative)
- `sparsity`          — sqrt(m0*m4) / m2
- `irregularity_factor` — m2^2 / (m0*m4), normalised by waveform length

Result: **360 features per gesture trial** (72 channels × 5 features).
All degenerate values (e.g. near-constant channels) are handled by
`safe_log` — clipped to [-10, 10] rather than blowing up to ±inf.

In [11]:
print("Computing Khushaba moments...")
start = time.time()

moments_df = create_khushaba_spectralmomentsFE_vectors(ppd_dict, fs=FS)

print(f"Done in {time.time()-start:.1f}s")
print(f"Shape: {moments_df.shape}   (expected: n_trials × 453 = n_trials × [3 meta + 450 features])")
moments_df.head()

Computing Khushaba moments...
Done in 42.9s
Shape: (3200, 363)   (expected: n_trials × 453 = n_trials × [3 meta + 450 features])


,Participant,Gesture_ID,Gesture_Num,IMU0_zero_order,IMU0_second_order,IMU0_fourth_order,IMU0_sparsity,IMU0_irregularity_factor,IMU1_zero_order,IMU1_second_order,...,IMU70_zero_order,IMU70_second_order,IMU70_fourth_order,IMU70_sparsity,IMU70_irregularity_factor,IMU71_zero_order,IMU71_second_order,IMU71_fourth_order,IMU71_sparsity,IMU71_irregularity_factor
0,P102,pan,1,-0.173894,-5.241832,-6.838296,1.735737,0.755370,0.858985,-7.404666,...,-2.454185,-1.145027,1.937089,0.886480,1.805018,-2.392227,-2.792539,0.173075,1.682964,1.822681
1,P102,pan,2,0.112834,-6.110651,-7.922873,2.205632,0.402352,1.137773,-7.459322,...,-2.224305,-1.601121,0.996678,0.987308,1.598510,-3.027316,-1.624369,2.811058,1.516249,1.978976
2,P102,pan,3,0.136351,-6.104009,-8.364103,1.990133,0.693293,1.100726,-7.965584,...,-2.137887,-1.184945,1.061258,0.646630,1.728467,-3.096748,-1.700275,2.588689,1.446257,2.194764
3,P102,pan,4,-0.091537,-5.714091,-7.589729,1.873458,0.764059,0.993344,-7.870080,...,-1.360822,-2.516870,-1.956760,0.858079,1.456771,-2.767322,-2.352683,1.327163,1.632606,2.022740
4,P102,pan,5,-0.041221,-5.764972,-7.606966,1.940878,0.669797,1.038775,-7.888794,...,-1.181768,-3.043325,-2.935941,0.984470,1.370833,-2.817178,-2.198381,1.562940,1.571265,1.980696


In [12]:
# Check for any surviving NaN or inf (should be zero with safe_log)
n_nan = moments_df.isna().sum().sum()
n_inf = np.isinf(moments_df.select_dtypes(include=np.number).values).sum()
print(f"NaN count: {n_nan}  |  Inf count: {n_inf}")

if n_nan > 0 or n_inf > 0:
    print("WARNING: Unexpected NaN/inf — imputing with column mean as fallback.")
    moments_df.replace([np.inf, -np.inf], np.nan, inplace=True)
    feature_cols = [c for c in moments_df.columns if c not in ['Participant','Gesture_ID','Gesture_Num']]
    moments_df[feature_cols] = moments_df[feature_cols].fillna(moments_df[feature_cols].mean())
else:
    print("Clean — no NaN or inf values.")

NaN count: 0  |  Inf count: 0
Clean — no NaN or inf values.


## 4. Global feature-matrix normalisation

Divides every feature by the global standard deviation of the entire
feature matrix, so all 450 features are on the same scale.
This is the same final step as in the EMG pipeline.

In [13]:
n_moments_df = normalize_whole_dataset_features(moments_df)

print(f"Shape after normalisation: {n_moments_df.shape}")
n_moments_df.head()

Shape after normalisation: (3200, 363)


,Participant,Gesture_ID,Gesture_Num,IMU0_zero_order,IMU0_second_order,IMU0_fourth_order,IMU0_sparsity,IMU0_irregularity_factor,IMU1_zero_order,IMU1_second_order,...,IMU70_zero_order,IMU70_second_order,IMU70_fourth_order,IMU70_sparsity,IMU70_irregularity_factor,IMU71_zero_order,IMU71_second_order,IMU71_fourth_order,IMU71_sparsity,IMU71_irregularity_factor
0,P102,pan,1,-0.041708,-1.257251,-1.640162,0.416316,0.181175,0.206027,-1.776006,...,-0.588635,-0.274634,0.464610,0.212622,0.432933,-0.573774,-0.669789,0.041512,0.403658,0.437169
1,P102,pan,2,0.027063,-1.465637,-1.900297,0.529020,0.096504,0.272894,-1.789115,...,-0.533499,-0.384028,0.239053,0.236805,0.383402,-0.726100,-0.389604,0.674231,0.363672,0.474657
2,P102,pan,3,0.032704,-1.464044,-2.006126,0.477332,0.166286,0.264009,-1.910541,...,-0.512771,-0.284208,0.254542,0.155094,0.414572,-0.742753,-0.407810,0.620896,0.346884,0.526413
3,P102,pan,4,-0.021955,-1.370522,-1.820393,0.449348,0.183259,0.238253,-1.887635,...,-0.326393,-0.603670,-0.469328,0.205810,0.349406,-0.663741,-0.564290,0.318319,0.391580,0.485153
4,P102,pan,5,-0.009887,-1.382726,-1.824527,0.465519,0.160651,0.249150,-1.892123,...,-0.283447,-0.729940,-0.704184,0.236125,0.328794,-0.675699,-0.527281,0.374870,0.376867,0.475069


## 5. Save

In [14]:
n_moments_df.to_csv(MOMENTS_CSV_SAVE_PATH, index=False, header=True)
print(f"Saved → {MOMENTS_CSV_SAVE_PATH}")
print(f"Final shape: {n_moments_df.shape}")
print(f"Columns: {list(n_moments_df.columns[:6])} ... {list(n_moments_df.columns[-3:])}")

Saved → C:\Users\kdmen\Box\Yamagami Lab\Data\Meta_Gesture_Project\moments_segraw_allIMU.csv
Final shape: (3200, 363)
Columns: ['Participant', 'Gesture_ID', 'Gesture_Num', 'IMU0_zero_order', 'IMU0_second_order', 'IMU0_fourth_order'] ... ['IMU71_fourth_order', 'IMU71_sparsity', 'IMU71_irregularity_factor']


## 6. Quick verification

In [15]:
# Reload and spot-check
check_df = pd.read_csv(MOMENTS_CSV_SAVE_PATH)
print(f"Reloaded shape : {check_df.shape}")
print(f"Participants   : {sorted(check_df['Participant'].unique())}")
print(f"Gestures       : {sorted(check_df['Gesture_ID'].unique())}")
print(f"Trials per gesture per participant (should be 10):")
print(check_df.groupby(['Participant','Gesture_ID'])['Gesture_Num'].count().describe())

feature_cols = [c for c in check_df.columns if c not in ['Participant','Gesture_ID','Gesture_Num']]
print(f"\nFeature stats (post-normalisation):")
print(check_df[feature_cols].describe().loc[['mean','std','min','max']])

Reloaded shape : (3200, 363)
Participants   : ['P004', 'P005', 'P006', 'P008', 'P010', 'P011', 'P102', 'P103', 'P104', 'P105', 'P106', 'P107', 'P108', 'P109', 'P110', 'P111', 'P112', 'P114', 'P115', 'P116', 'P118', 'P119', 'P121', 'P122', 'P123', 'P124', 'P125', 'P126', 'P127', 'P128', 'P131', 'P132']
Gestures       : ['close', 'delete', 'duplicate', 'move', 'open', 'pan', 'rotate', 'select-single', 'zoom-in', 'zoom-out']
Trials per gesture per participant (should be 10):
count    320.0
mean      10.0
std        0.0
min       10.0
25%       10.0
50%       10.0
75%       10.0
max       10.0
Name: Gesture_Num, dtype: float64

Feature stats (post-normalisation):
      IMU0_zero_order  IMU0_second_order  IMU0_fourth_order  IMU0_sparsity   
mean        -0.361574          -0.491267          -0.114095       0.267014  \
std          0.399989           0.634249           1.282434       0.142215   
min         -1.979304          -2.398495          -2.398495       0.041915   
max          0.62466